# 🌿 Plant Disease Scanner — Model Training
**Architecture:** EfficientNetV2-S with transfer learning  
**Dataset:** PlantVillage (38 classes, ~54,000 images)  
**Expected accuracy:** ~96%+ val accuracy  
**GPU:** T4 recommended (free on Colab) — ~25 min training

## Steps:
1. Set runtime to GPU: `Runtime → Change runtime type → T4 GPU`
2. Run all cells in order
3. Download `tfjs_model.zip` at the end
4. Extract into `plant-disease-scanner/public/model/`

In [ ]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────────
import tensorflow as tf
print(f'TensorFlow: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs: {gpus}')
if not gpus:
    print('⚠ No GPU detected! Go to Runtime → Change runtime type → T4 GPU')
else:
    print(f'✓ GPU ready: {gpus[0].name}')

In [ ]:
# ── Cell 2: Install TF.js converter ───────────────────────────────────
!pip install tensorflowjs -q
import tensorflowjs as tfjs
print(f'TF.js converter: {tfjs.__version__}')

In [ ]:
# ── Cell 3: Download PlantVillage dataset from Kaggle ──────────────────
# Option A: Kaggle API (recommended)
# 1. Go to https://www.kaggle.com/settings → API → Create New Token
# 2. Upload the downloaded kaggle.json when prompted below

from google.colab import files
print('Upload your kaggle.json file:')
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d emmarex/plantdisease
!unzip -q plantdisease.zip -d ./PlantVillage_raw

import os
# Find the actual data directory
for root, dirs, files_list in os.walk('./PlantVillage_raw'):
    if any(d.startswith('Apple___') or d.startswith('Tomato___') for d in dirs):
        DATA_DIR = root
        print(f'Dataset found at: {DATA_DIR}')
        print(f'Classes: {len(dirs)}')
        break

In [ ]:
# ── Cell 3B (Alternative): Use pre-split dataset from Drive ───────────
# If you already have the dataset on Google Drive, mount it:
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/PlantVillage'

# Or if you ran Cell 3, just verify:
import os
classes = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]
print(f'Total classes: {len(classes)}')
total_images = sum(len(os.listdir(os.path.join(DATA_DIR, c))) for c in classes)
print(f'Total images: {total_images:,}')
print('Sample classes:', classes[:5])

In [ ]:
# ── Cell 4: Configuration ──────────────────────────────────────────────
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow as tf

IMG_SIZE      = 224
BATCH_SIZE    = 64    # Larger batch on GPU
PHASE1_EPOCHS = 5     # Train head only
PHASE2_EPOCHS = 20    # Fine-tune backbone
LR_HEAD       = 1e-3
LR_FINETUNE   = 5e-5

print(f'Config ready. Image: {IMG_SIZE}x{IMG_SIZE}, Batch: {BATCH_SIZE}')

In [ ]:
# ── Cell 5: Load dataset ───────────────────────────────────────────────
train_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset='training',
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical',
)

val_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset='validation',
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical',
)

CLASS_NAMES = train_ds.class_names
NUM_CLASSES = len(CLASS_NAMES)
print(f'Classes: {NUM_CLASSES}')
print(f'Train batches: {len(train_ds)}')
print(f'Val batches: {len(val_ds)}')
print('Classes:', CLASS_NAMES[:5], '...')

In [ ]:
# ── Cell 6: Data augmentation pipeline ────────────────────────────────
# Aggressive augmentation mimics real field conditions:
# variable lighting, camera angles, distances
augment = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.2),
    layers.RandomZoom((-0.25, 0.2)),
    layers.RandomBrightness(0.25),
    layers.RandomContrast(0.25),
], name='augmentation')

AUTOTUNE = tf.data.AUTOTUNE

train_ds = (
    train_ds
    .map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)
val_ds = val_ds.prefetch(AUTOTUNE)
print('Augmentation pipeline ready')

In [ ]:
# ── Cell 7: Build EfficientNetV2-S model ──────────────────────────────
# EfficientNetV2-S pretrained on ImageNet21k
# Better accuracy than EfficientNetB0 especially on small datasets

def build_model(num_classes, dropout=0.3):
    # Input
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='input_image')
    
    # Backbone: EfficientNetV2-S (frozen initially)
    backbone = keras.applications.EfficientNetV2S(
        include_top=False,
        weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_preprocessing=True,
    )
    backbone.trainable = False
    
    x = backbone(inputs, training=False)
    
    # Classification head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(512, activation='swish',
                     kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(
        num_classes, activation='softmax',
        kernel_regularizer=keras.regularizers.l2(1e-4)
    )(x)
    
    return keras.Model(inputs, outputs, name='PlantDiseaseClassifier')


model = build_model(NUM_CLASSES)

trainable = sum(np.prod(w.shape) for w in model.trainable_weights)
total = sum(np.prod(w.shape) for w in model.weights)
print(f'Total params:     {total:,}')
print(f'Trainable params: {trainable:,} (head only)')

In [ ]:
# ── Cell 8: Phase 1 — Train head only ─────────────────────────────────
model.compile(
    optimizer=keras.optimizers.Adam(LR_HEAD),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3, name='top3')],
)

callbacks_p1 = [
    keras.callbacks.ModelCheckpoint('best_model.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor='val_accuracy',
        patience=3, restore_best_weights=True),
]

print(f'Phase 1: Training classification head for {PHASE1_EPOCHS} epochs...')
history1 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=PHASE1_EPOCHS,
    callbacks=callbacks_p1,
)

print(f"\nPhase 1 best val accuracy: {max(history1.history['val_accuracy']):.4f}")

In [ ]:
# ── Cell 9: Phase 2 — Fine-tune top 30% of backbone ───────────────────
backbone_layer = model.get_layer('efficientnetv2-s')
total_layers = len(backbone_layer.layers)
unfreeze_from = int(total_layers * 0.7)  # unfreeze top 30%

backbone_layer.trainable = True
for i, layer in enumerate(backbone_layer.layers):
    layer.trainable = (i >= unfreeze_from)

frozen   = sum(1 for l in backbone_layer.layers if not l.trainable)
unfrozen = sum(1 for l in backbone_layer.layers if l.trainable)
print(f'Backbone: {frozen} frozen, {unfrozen} unfrozen layers')

model.compile(
    optimizer=keras.optimizers.Adam(LR_FINETUNE),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3, name='top3')],
)

callbacks_p2 = [
    keras.callbacks.ModelCheckpoint('best_model.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor='val_accuracy',
        patience=6, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss',
        factor=0.5, patience=3, min_lr=1e-7, verbose=1),
]

print(f'Phase 2: Fine-tuning for {PHASE2_EPOCHS} epochs...')
history2 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=PHASE1_EPOCHS + PHASE2_EPOCHS,
    initial_epoch=PHASE1_EPOCHS,
    callbacks=callbacks_p2,
)

best = max(history2.history['val_accuracy'])
top3 = max(history2.history['val_top3'])
print(f'\n✅ Best val accuracy: {best:.4f}')
print(f'   Best val top-3:    {top3:.4f}')

In [ ]:
# ── Cell 10: Plot training curves ─────────────────────────────────────
import matplotlib.pyplot as plt

all_acc     = history1.history['accuracy']     + history2.history['accuracy']
all_val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
all_loss    = history1.history['loss']         + history2.history['loss']
all_val_loss= history1.history['val_loss']     + history2.history['val_loss']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(all_acc, label='Train', color='#2D5016', linewidth=2)
ax1.plot(all_val_acc, label='Validation', color='#C17E2D', linewidth=2)
ax1.axvline(PHASE1_EPOCHS, color='gray', linestyle='--', alpha=0.5, label='Fine-tune start')
ax1.set_title('Accuracy', fontsize=14)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(all_loss, label='Train', color='#2D5016', linewidth=2)
ax2.plot(all_val_loss, label='Validation', color='#C17E2D', linewidth=2)
ax2.axvline(PHASE1_EPOCHS, color='gray', linestyle='--', alpha=0.5)
ax2.set_title('Loss', fontsize=14)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved')

In [ ]:
# ── Cell 11: Confusion matrix on validation set ────────────────────────
from sklearn.metrics import classification_report
import numpy as np

y_true, y_pred = [], []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))

In [ ]:
# ── Cell 12: Export SavedModel ─────────────────────────────────────────
import json, os

os.makedirs('./output', exist_ok=True)
model.export('./output/saved_model')
print('✓ SavedModel exported')

# Save class names index
model_info = {
    'num_classes': NUM_CLASSES,
    'class_names': CLASS_NAMES,
    'input_size': IMG_SIZE,
    'input_range': [0, 255],
    'architecture': 'EfficientNetV2-S',
    'val_accuracy': float(max(history2.history['val_accuracy'])),
    'val_top3': float(max(history2.history['val_top3'])),
}
with open('./output/model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)
print('✓ model_info.json saved')
print(json.dumps(model_info, indent=2))

In [ ]:
# ── Cell 13: Convert to TF.js with float16 quantization ───────────────
# Float16 cuts model size ~50% with <0.5% accuracy loss
!tensorflowjs_converter \
    --input_format=tf_saved_model \
    --output_format=tfjs_graph_model \
    --signature_name=serving_default \
    --saved_model_tags=serve \
    --quantize_float16='*' \
    ./output/saved_model \
    ./output/tfjs_model

# Show output files and sizes
import os
total_size = 0
for f in sorted(os.listdir('./output/tfjs_model')):
    size = os.path.getsize(f'./output/tfjs_model/{f}')
    total_size += size
    print(f'  {f}: {size/1024:.1f} KB')
print(f'\nTotal model size: {total_size/1024/1024:.1f} MB')
print('✅ TF.js model ready!')

In [ ]:
# ── Cell 14: Test TF.js model prediction ──────────────────────────────
# Verify the converted model gives same predictions as the Keras model
import tensorflowjs as tfjs
import numpy as np

# Load tfjs model back for verification
# (In practice the frontend does this — this is just a sanity check)
test_images, test_labels = next(iter(val_ds.take(1)))
keras_preds = model.predict(test_images[:4], verbose=0)

for i in range(4):
    true_class = CLASS_NAMES[np.argmax(test_labels[i])]
    pred_class = CLASS_NAMES[np.argmax(keras_preds[i])]
    confidence = np.max(keras_preds[i])
    correct = '✓' if true_class == pred_class else '✗'
    print(f'{correct} True: {true_class[:40]:40s} | Pred: {pred_class[:40]:40s} ({confidence:.1%})')

In [ ]:
# ── Cell 15: Download model ────────────────────────────────────────────
import shutil
from google.colab import files

# Zip the TF.js model + model_info.json
shutil.make_archive('tfjs_model', 'zip', './output/tfjs_model')
shutil.copy('./output/model_info.json', './model_info.json')

print('Downloading tfjs_model.zip...')
files.download('tfjs_model.zip')
files.download('model_info.json')
files.download('training_curves.png')

print('\n✅ Download complete!')
print('\nNext steps:')
print('1. Extract tfjs_model.zip into plant-disease-scanner/public/model/')
print('2. Copy model_info.json to plant-disease-scanner/public/model/')
print('3. Run: npm run build && git push')